# Coding Agent from scratch — Together AI + Eval

This notebook uses your **existing uv project** and Jupyter kernel. No `mini-swe-agent` framework, no fake LLM.

Install in your existing project's terminal: `uv add together`. Select the project's Python kernel in JupyterLab.

Set `TOGETHER_API_KEY` in the environment *before launching JupyterLab*. Avoid storing API keys in notebooks.

We will build a 3-tool coding agent (read, write, test) and then evaluate it using unseen tests. Execution uses a local subprocess, **not a security sandbox**; use only for controlled exercises. We deliberately do not grant arbitrary shell access.

In [4]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from together import Together

load_dotenv(Path.cwd() / ".env", override=True)

api_key = os.getenv("TOGETHER_API_KEY")
assert api_key, "TOGETHER_API_KEY not found"

client = Together(api_key=api_key)

print("Together API ready!")

Together API ready!


In [3]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "What is 1 + 1?"}
    ],
)

print(response.choices[0].message.content)

1 + 1 = 2.


## 1 — Create a buggy coding task

A scratch directory is created automatically. This does **not** create another uv project or modify your existing code. Public tests are visible to the agent, hidden tests will not be exposed.

In [5]:
workspace = Path(tempfile.mkdtemp(prefix="agent-tutorial-"))

(workspace / "solution.py").write_text("""def median(values):
    values = sorted(values)
    if not values:
        raise ValueError('empty input')
    return values[len(values) // 2]
""", encoding="utf-8")

(workspace / "test_public.py").write_text("""import unittest
from solution import median

class TestMedian(unittest.TestCase):
    def test_odd(self):
        self.assertEqual(median([3, 1, 2]), 2)

    def test_even(self):
        self.assertEqual(median([1, 3, 5, 7]), 4)
""", encoding="utf-8")

print("Scratch workspace:", workspace)
print((workspace / "solution.py").read_text())

Scratch workspace: /var/folders/t6/m2jnmr7967x46bg79pfng_bm0000gn/T/agent-tutorial-978l97dr
def median(values):
    values = sorted(values)
    if not values:
        raise ValueError('empty input')
    return values[len(values) // 2]



## 2 — Build the evaluator first

We run tests in a subprocess so that a crash does not terminate Jupyter. **This is not process or filesystem isolation**. We strip API credentials from the child process environment. Later we can replace this with Docker.

In [6]:
def run_tests(folder: Path, pattern: str) -> dict:
    proc = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", str(folder), "-p", pattern, "-v"],
        cwd=folder,
        env={"PATH": os.environ.get("PATH", ""), "PYTHONIOENCODING": "utf-8"},
        capture_output=True,
        text=True,
        timeout=10,
    )
    return {"passed": proc.returncode == 0, "output": (proc.stdout + proc.stderr)[-3000:]}

baseline = run_tests(workspace, "test_public.py")
print("Baseline passed:", baseline["passed"])
print(baseline["output"])

Baseline passed: False
test_even (test_public.TestMedian.test_even) ... FAIL
test_odd (test_public.TestMedian.test_odd) ... ok

FAIL: test_even (test_public.TestMedian.test_even)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/var/folders/t6/m2jnmr7967x46bg79pfng_bm0000gn/T/agent-tutorial-978l97dr/test_public.py", line 9, in test_even
    self.assertEqual(median([1, 3, 5, 7]), 4)
AssertionError: 5 != 4

----------------------------------------------------------------------
Ran 2 tests in 0.000s

FAILED (failures=1)



## 3 — Define tools

The LLM can only ask to read two known files, overwrite `solution.py`, or run public tests. These are **our tools**, not `mini-swe-agent` tools.

In [8]:
TOOLS = [
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file in the task workspace.",
        "parameters": {"type": "object", "properties": {
            "filename": {"type": "string", "enum": ["solution.py", "test_public.py"]}},
            "required": ["filename"], "additionalProperties": False}}},
    
    {"type": "function", "function": {
        "name": "write_solution", "description": "Replace solution.py with complete Python code.",
        "parameters": {"type": "object", "properties": {
            "content": {"type": "string"}},
            "required": ["content"], "additionalProperties": False}}},
    
    {"type": "function", "function": {
        "name": "run_public_tests", "description": "Run tests visible to the agent.",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
]

def execute_tool(name: str, args: dict) -> dict:
    if name == "read_file":
        filename = args["filename"]
        
        if filename not in {"solution.py", "test_public.py"}:
            raise ValueError("File not allowed")
            
        return {"content": (workspace / filename).read_text(encoding="utf-8")}
        
    if name == "write_solution":
        content = args["content"]
        if not isinstance(content, str) or len(content) > 20000:
            raise ValueError("Invalid content")
            
        (workspace / "solution.py").write_text(content, encoding="utf-8")
        return {"status": "written"}
        
    if name == "run_public_tests":
        return run_tests(workspace, "test_public.py")
        
    raise ValueError(f"Unknown tool: {name}")

## 4 — Real agent loop

Watch the messages and tool calls. This is the mechanism: **LLM → tool call → observation → LLM**. No agent framework.

In [10]:
MODEL = "openai/gpt-oss-120b"

SYSTEM = """You are a coding agent. Your job is to fix solution.py.
Tools available: read_file, write_solution, run_public_tests.
Read source and tests first, implement a general solution, and run public tests.
Do not hardcode individual test values. When done, give a short final answer.
"""

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Fix median() for any list of numbers, preserving the ValueError for empty lists."},
]

MAX_STEPS = 12
token_usage = 0

for step in range(1, MAX_STEPS + 1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
        max_tokens=2000,
    )
    
    token_usage += getattr(response.usage, "total_tokens", 0) or 0
    
    msg = response.choices[0].message
    calls = msg.tool_calls or []
    assistant_message = {"role": "assistant", "content": msg.content or ""}
    
    if calls:
        assistant_message["tool_calls"] = [
            {"id": call.id, "type": "function", "function": {
                "name": call.function.name, "arguments": call.function.arguments}}
            for call in calls
        ]
        
    messages.append(assistant_message)

    print(f"\nSTEP {step}: {msg.content or '(tool call)'}")
    
    if not calls:
        break

    for call in calls:
        args = {}
        
        try:
            args = json.loads(call.function.arguments or "{}")
            result = execute_tool(call.function.name, args)
        except Exception as error:
            result = {"error": f"{type(error).__name__}: {error}"}
            
        print("TOOL:", call.function.name, "ARGS:", list(args))
        print("RESULT:", str(result)[:800])
        
        messages.append({
            "role": "tool", "tool_call_id": call.id,
            "content": json.dumps(result, ensure_ascii=False),
        })
else:
    print("Reached MAX_STEPS; agent stopped.")

print("\nTotal tokens (reported by API):", token_usage)
print("\nFinal solution:\n", (workspace / "solution.py").read_text())


STEP 1: Implemented a robust `median` function in **solution.py**:
- Handles empty input by raising `ValueError`.
- Returns the middle element for odd-length lists.
- Computes the average of the two central elements for even-length lists.
- Does not mutate the original input sequence.
TOOL: read_file ARGS: ['filename']
RESULT: {'content': "def median(values):\n    values = sorted(values)\n    if not values:\n        raise ValueError('empty input')\n    return values[len(values) // 2]\n"}
TOOL: read_file ARGS: ['filename']
RESULT: {'content': "def median(values):\n    values = sorted(values)\n    if not values:\n        raise ValueError('empty input')\n    return values[len(values) // 2]\n"}
TOOL: read_file ARGS: ['filename']
RESULT: {'content': 'import unittest\nfrom solution import median\n\nclass TestMedian(unittest.TestCase):\n    def test_odd(self):\n        self.assertEqual(median([3, 1, 2]), 2)\n\n    def test_even(self):\n        self.assertEqual(median([1, 3, 5, 7]), 4)\n'}
TO

## 5 — Independent hidden evaluation

This evaluation is outside the agent workspace; its tests were **not included in the model's tool outputs**. The code is copied to a separate temporary evaluation directory.

In [ ]:
hidden_tests = """import unittest
from solution import median

class HiddenTests(unittest.TestCase):
    def test_negative_even(self):
        self.assertEqual(median([-7, -1, 3, 5]), 1)

    def test_duplicates(self):
        self.assertEqual(median([9, 1, 1, 1]), 1)

    def test_float(self):
        self.assertEqual(median([0.1, 0.3]), 0.2)

    def test_single(self):
        self.assertEqual(median([42]), 42)

    def test_even_unsorted(self):
        self.assertEqual(median([8, 1, 2, 7]), 4.5)

    def test_empty(self):
        with self.assertRaises(ValueError):
            median([])
"""

eval_dir = Path(tempfile.mkdtemp(prefix="agent-hidden-eval-"))
shutil.copy2(workspace / "solution.py", eval_dir / "solution.py")
(eval_dir / "test_hidden.py").write_text(hidden_tests, encoding="utf-8")

public = run_tests(workspace, "test_public.py")
hidden = run_tests(eval_dir, "test_hidden.py")
print("Public PASS:", public["passed"])
print("Hidden PASS:", hidden["passed"])
print("Hidden details:", hidden["output"])
print("Agent steps:", min(step, MAX_STEPS))

### Next experiments

1. Change `MODEL` and compare public vs hidden pass rate.
2. Rerun from a fresh workspace with identical tasks 5 times: estimate variance.
3. Remove `run_public_tests` tool and ask whether iterative feedback improves success.
4. Move test execution into Docker before using untrusted repositories.

**Important:** The LLM is real; the notebook's API calls have **not** been executed here (no access to your key). You can run cells in order in your own kernel.